Генерация кварталов

In [ ]:
!pip install blocksnet ipykernel -q

In [ ]:
!pip install folium matplotlib mapclassify

In [ ]:
import osmnx as ox
import geopandas as gpd
import os

In [5]:
boundary = ox.geocode_to_gdf('Санкт-Петербург')

In [7]:
tags = {
    'roads': {
      "highway": ["construction","crossing","living_street","motorway","motorway_link","motorway_junction","pedestrian","primary","primary_link","raceway","residential","road","secondary","secondary_link","services","tertiary","tertiary_link","track","trunk","trunk_link","turning_circle","turning_loop","unclassified",],
      "service": ["living_street", "emergency_access"]
    },
    'railways': {
      "railway": "rail"
    },
    'water': {
      'riverbank':True,
      'reservoir':True,
      'basin':True,
      'dock':True,
      'canal':True,
      'pond':True,
      'natural':['water','bay'],
      'waterway':['river','canal','ditch'],
      'landuse':'basin',
      'water': 'lake'
    }
}

In [8]:
import geopandas as gpd

water = ox.features_from_polygon(boundary.unary_union, tags['water'])
roads = ox.features_from_polygon(boundary.unary_union, tags['roads'])
railways = ox.features_from_polygon(boundary.unary_union, tags['railways'])

In [9]:
local_crs = boundary.estimate_utm_crs()

In [10]:
boundary = boundary.reset_index()[['geometry']].to_crs(local_crs)
water = water.reset_index()[['geometry']].to_crs(local_crs)
roads = roads.reset_index()[['geometry']].to_crs(local_crs)
railways = railways.reset_index()[['geometry']].to_crs(local_crs)

In [11]:
roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])]

In [ ]:
import momepy
GAP_TOLERANCE = 1
def _get_roads(roads):
    merged = roads.unary_union
    if merged.geom_type == 'MultiLineString':
        roads = gpd.GeoDataFrame(geometry=list(merged.geoms), crs=roads.crs)
    else:
        roads = gpd.GeoDataFrame(geometry=[merged], crs=roads.crs)
    roads = roads.explode(index_parts=False).reset_index(drop=True)
    roads.geometry = momepy.close_gaps(roads, GAP_TOLERANCE)
    roads = roads[roads.geom_type.isin(['LineString'])]
    return roads

roads = _get_roads(roads)
roads


In [ ]:
from blocksnet import BlocksGenerator

bg = BlocksGenerator(boundary, roads, None, water)

In [ ]:
blocks = bg.run()

In [16]:
blocks.to_file('/Users/polina/Downloads/Казань_Аня/blocks_Kazan.geojson')

In [ ]:
!pip install mapclassify -q

In [16]:
import geopandas as gpd
buildings = ox.features_from_polygon(boundary.to_crs(4326).unary_union, {'building': True})

In [17]:
buildings = buildings.to_crs(local_crs).reset_index()[['geometry']]
buildings.geometry = buildings.representative_point()

In [18]:
from blocksnet import BlocksSplitter

bs = BlocksSplitter(blocks, buildings)

In [ ]:
splitted_blocks = bs.run()

In [ ]:
len(blocks), len(splitted_blocks)

In [ ]:
blocks.plot(linewidth=0.1, figsize=(10,10)).set_axis_off()
splitted_blocks.plot(linewidth=0.1, figsize=(10,10)).set_axis_off()

In [46]:
splitted_blocks.to_file('/Users/polina/Downloads/splitted_blocks_SPB_Polina.geojson')